## 1. Installing Required Python Dependencies

This section installs all required libraries and frameworks needed for the Agentic Cyber Security Assistant, including LangChain components, Pinecone integration, PDF processing, embedding models, and environment configuration utilities.


In [1]:
!pip install -U \
langchain \
langchain-community \
langchain-text-splitters \
langchain-huggingface \
langchain-pinecone \
pinecone \
pypdf \
sentence-transformers \
python-dotenv

  Obtaining dependency information for pinecone from https://files.pythonhosted.org/packages/c5/e8/4f4e6570650904ff71383b404e667382c92d54b31c5f56a4f037cc17c57f/pinecone-9.1.0-cp310-abi3-win_amd64.whl.metadata
  Obtaining dependency information for sentence-transformers from https://files.pythonhosted.org/packages/c1/ad/8f73f512dc7ad4031d2b64cbb67f70bdfb355756afbe0db610a5146415c1/sentence_transformers-5.6.1-py3-none-any.whl.metadata
  Obtaining dependency information for msgspec from https://files.pythonhosted.org/packages/cc/4d/619866af2840875be408047bf9e70ceafbae6ab50660de7134ed1b25eb86/msgspec-0.21.1-cp312-cp312-win_amd64.whl.metadata
  Obtaining dependency information for h2<5,>=3 from https://files.pythonhosted.org/packages/f6/df/5b14a118322d6097cb9bb30ec6bacad268e546a8ecfcb1f6d0de618dac2f/h2-4.4.0-py3-none-any.whl.metadata
  Obtaining dependency information for hyperframe<7,>=6.1 from https://files.pythonhosted.org/packages/48/30/47d0bf6072f7252e6521f3447ccfa40b421b6824517f8285470


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Importing Required Libraries and Framework Components

This section imports the core libraries


In [3]:
import os

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from pinecone import Pinecone, ServerlessSpec

from langchain_pinecone import PineconeVectorStore


C:\Users\user\AppData\Local\Temp\ipykernel_9156\1243944404.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## 3. Configuring Pinecone API Authentication

This section configures the Pinecone API key required to connect the application with the Pinecone vector database.
Masked the API key
Loads the CIS PDF

In [4]:
PINECONE_API_KEY = "pcsk_68NMyM_QSqGSsYYyPVTA8kckk2VYZdJ2UKQaGdwU5evYSqmid4iG8V495E91uV8BGosH4U" 

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY


In [5]:
PDF_PATH = r"C:\Users\user\Desktop\Shubham-AI\Shared\ey-ai-upskill-10-main\ey-ai-upskill-10-main\16-rag-capstone\cis-basic\data\cis_docs\CIS_Microsoft_Windows_11_Enterprise_Benchmark_v5.0.1.pdf"

## 4. Loading Cybersecurity Knowledge Base Documents

This section loads the PDF-based cybersecurity knowledge source using the PyPDFLoader.



In [6]:
loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

print("Number of pages:", len(documents))


Number of pages: 1454


## 5. Splitting Documents into Manageable Text Chunks

This section divides the loaded cybersecurity documents into smaller text chunks using a recursive text splitter.

Chunking improves the efficiency of semantic search by creating smaller, meaningful sections that can be converted into embeddings and retrieved accurately from the vector database during RAG-based question answering.

Configuration:
- **Chunk size:** 1000 characters
- **Chunk overlap:** 200 characters to preserve context between consecutive chunks


In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))


Total chunks: 2674


# Install Required Python Libraries

This cell installs all required dependencies for the Cyber Security Assistant project, including LangChain, Pinecone, PDF processing, embedding models, and environment management libraries.


In [8]:
for chunk in chunks:
    chunk.metadata["source"] = "CIS Windows 11 Benchmark"
    chunk.metadata["document"] = "CIS_Microsoft_Windows_11_Enterprise_Benchmark_v5.0.1"


# Initialize HuggingFace Embedding Model

This cell initializes the HuggingFace embedding model used to convert document text chunks into numerical vector representations. These embeddings are used for similarity search and retrieval from the Pinecone vector database.


In [9]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# Test Embedding Generation

This cell validates the HuggingFace embedding model by generating an embedding vector for a sample security-related query. It checks that the model is working correctly and displays the size of the generated vector representation.


In [10]:
test_vector = embedding_model.embed_query(
    "Windows password policy requirements"
)

print(len(test_vector))


384


# Create and Configure Pinecone Vector Database Index

This cell initializes the Pinecone client, checks whether the required vector index already exists, and creates a new index if needed. The index is configured to store embedding vectors generated by the HuggingFace model using cosine similarity for document retrieval.


In [11]:
INDEX_NAME = "cyber-security"


pc = Pinecone(
    api_key=PINECONE_API_KEY
)


existing_indexes = [
    index.name 
    for index in pc.list_indexes()
]


if INDEX_NAME not in existing_indexes:

    pc.create_index(
        name=INDEX_NAME,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )


print("Index ready")


Index ready


# Initialize Pinecone Vector Store Connection

This cell connects to the created Pinecone index and initializes the vector store using the HuggingFace embedding model. The configured namespace is used to organize and retrieve CIS benchmark documents stored in the vector database.


In [12]:
NAMESPACE = "cis_documents"


index = pc.Index(INDEX_NAME)


vectorstore = PineconeVectorStore(
    index=index,
    embedding=embedding_model,
    namespace=NAMESPACE
)


# Upload Document Chunks to Pinecone Vector Database

This cell uploads the processed CIS benchmark document chunks into the Pinecone vector store. Each chunk is converted into an embedding vector and stored for efficient similarity-based retrieval during RAG (Retrieval-Augmented Generation) queries.


In [13]:
vectorstore.add_documents(
    documents=chunks
)


print("CIS documents uploaded successfully")


CIS documents uploaded successfully


# Configure Pinecone Retriever

This cell creates a retriever from the Pinecone vector store.  
The retriever is configured to perform similarity search and return the top 3 most relevant document chunks for a given cybersecurity query.


In [14]:
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 3
    }
)


# Test CIS Benchmark Document Retrieval

This cell tests the RAG retrieval pipeline by sending a cybersecurity query to the Pinecone retriever.

The retriever searches the vector database and returns the most relevant CIS benchmark document chunks related to securing Windows SMB.


In [15]:
results = retriever.invoke(
    "How should Windows SMB be secured?"
)


for doc in results:
    print("----------------")
    print(doc.page_content[:500])


----------------
the threat actor could pose as the server or client after legitimate authentication and 
gain unauthorized access to data. 
SMB is the resource sharing protocol that is supported by many Windows operating 
systems. It is the basis of NetBIOS and many other protocols. SMB signatures 
authenticate both users and the servers that host the data. If either side fails the 
authentication process, data transmission will not take place. 
Impact: 
The Microsoft network client will not communicate with a 
----------------
the threat actor could pose as the server or client after legitimate authentication and 
gain unauthorized access to data. 
SMB is the resource sharing protocol that is supported by many Windows operating 
systems. It is the basis of NetBIOS and many other protocols. SMB signatures 
authenticate both users and the servers that host the data. If either side fails the 
authentication process, data transmission will not take place. 
Impact: 
The Microsoft network 

In [16]:
print("Moving from RAG setup to Agent setup...")


Moving from RAG setup to Agent setup...


In [17]:
!pip install -U langchain-groq



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Import Agent Framework and Workflow Dependencies

This cell imports the required libraries for building the Agentic Cyber Security Assistant.



In [18]:
import requests
import json

from typing import TypedDict

from langchain.tools import tool
from langchain.agents import create_agent

from langchain_groq import ChatGroq

from langgraph.graph import StateGraph, START, END


# Configure API Keys and Project Environment Variables

This cell configures the authentication keys and environment settings required by the Agentic Cyber Security Assistant.
MAsking done for security purpose.


In [19]:
GROQ_API_KEY = "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"

PINECONE_API_KEY = "pcsk_68NMyM_QSqGSsYYyPVTA8kckk2VYZdJ2UKQaGdwU5evYSqmid4iG8V495E91uV8BGosH4U"

NVD_API_KEY = "6ced8a78-5c93-4c84-aad2-3959806c6074"


INDEX_NAME = "cyber-security"

NAMESPACE = "cis_documents"


os.environ["GROQ_API_KEY"] = GROQ_API_KEY

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY


## Configure the Groq Llama 3.1 Model for AI Agents


In [52]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    max_tokens=250

)

print("Groq LLM ready")


Groq LLM ready


## Connect to Pinecone and Create the RAG Retriever


In [53]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


pc = Pinecone(
    api_key=PINECONE_API_KEY
)


index = pc.Index(
    INDEX_NAME
)


vectorstore = PineconeVectorStore(
    index=index,
    embedding=embedding_model,
    namespace=NAMESPACE
)


retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k":3,
        "fetch_k":10
    }
)


print("Pinecone retriever ready")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Pinecone retriever ready


## Define the CIS Document Retrieval Tool


In [54]:
@tool
def ask_cis(question:str)->str:
    """Get CIS Windows hardening recommendations."""


    docs = retriever.invoke(question)

    if not docs:
        return "No CIS documents found."


    result = []

    for i,doc in enumerate(docs):

        result.append(
            f"""
Document {i+1}

{doc.page_content}
"""
        )


    return "\n".join(result[:2])



## Test the CIS Retrieval Tool with a Sample Security Query


In [55]:
print(
    ask_cis.invoke(
        "How should SMB signing be configured?"
    )
)



Document 1

Page 560 
18.6.7.2 Ensure 'Audit client does not support signing' is set to 
'Enabled' (Automated) 
Profile Applicability: 
•  Level 1 (L1) 
Description: 
This policy setting determines whether the Server Message Block (SMB) server will log 
events when the SMB client doesn't support signing. 
Enabling this will create event log entries in Applications and Services 
Logs\Microsoft\Windows\SMBClient\Audit, with Event ID 31999. 
The recommended state for this setting is: Enabled. 
Rationale: 
Organizations should be aware of all unsigned SMB traffic in their environment. Older 
SMB protocols that do not use signing can make an environment susceptible to many 
types of attacks, including SMB interception attacks. 
Impact: 
All SMB traffic that is unsigned will be logged as an event. 
Audit: 
Navigate to the UI Path articulated in the Remediation section and confirm it is set as 
prescribed. This group policy setting is backed by the following registry location with a


Docume

## Define Threat Intelligence Tools for CVE, CISA KEV, MITRE ATT&CK, and EPSS Lookups


In [87]:
@tool
def lookup_cve(keyword:str)->str:
    """
    Find CVE vulnerabilities.
    """

    url = (
        "https://services.nvd.nist.gov/rest/json/cves/2.0"
    )


    params = {
        "keywordSearch": keyword,
        "resultsPerPage":3
    }


    headers = {}

    if NVD_API_KEY:
        headers["apiKey"] = NVD_API_KEY


    try:

        response = requests.get(
            url,
            params=params,
            headers=headers,
            timeout=20
        )


        response.raise_for_status()


        data=response.json()


        results=[]


        for item in data.get(
            "vulnerabilities",
            []
        ):

            cve=item["cve"]

            results.append(
                f"""
{cve['id']}

{cve['descriptions'][0]['value'][:300]}
"""
            )


        return "\n".join(results[:3])



    except Exception as e:

        return str(e)
        
@tool
def lookup_cisa_kev(keyword: str) -> str:
    """
    Find actively exploited vulnerabilities.
    """

    url = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json"

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    data = response.json()

    results = []

    for item in data["vulnerabilities"]:

        text = (
            item["vendorProject"] +
            item["product"] +
            item["shortDescription"]
        ).lower()

        if keyword.lower() in text:

            results.append(
                f"""
CVE: {item['cveID']}

Vendor: {item['vendorProject']}

Product: {item['product']}

Description: {item['shortDescription']}

Required Action:
{item['requiredAction']}
"""
            )

    if results:
        return "\n\n".join(results[:2])


    return "No KEV entries found."

@tool
def lookup_attack(keyword: str) -> str:
    """
    Search MITRE ATT&CK techniques.
    """

    url = "https://raw.githubusercontent.com/mitre/cti/master/enterprise-attack/enterprise-attack.json"

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    data = response.json()

    results = []

    for obj in data["objects"]:

        if obj.get("type") != "attack-pattern":
            continue

        if keyword.lower() in obj.get("name", "").lower():

            results.append(
                f"""
Technique:
{obj['name']}

Description:
{obj.get('description', '')[:400]}
"""
            )

    if results:
        return "\n\n".join(results[:5])

    return "No ATT&CK technique found."

@tool
def lookup_epss(cve: str) -> str:
    """
    Retrieve the EPSS score for a CVE.
    """

    url = f"https://api.first.org/data/v1/epss?cve={cve}"

    response = requests.get(url, timeout=20)
    response.raise_for_status()

    data = response.json()

    if not data["data"]:
        return "No EPSS score available."

    result = data["data"][0]

    return f"""
CVE:
{result['cve']}

EPSS Score:
{result['epss']}

Percentile:
{result['percentile']}
"""


## Create the Planner Agent for Query Routing


In [88]:
planner_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="""
You are the Planner Agent.

Return JSON only.

{
"use_cyber": true/false,
"reason": ""
}

Rules:

- If the query is related to cybersecurity,
  set "use_cyber" to true.

- Otherwise set it to false.

Do not answer the question.
Only return valid JSON.
"""
)


In [42]:
#retrieval_agent=create_agent(
   # model=llm,
    #tools=[ask_cis],
    #system_prompt="""
#You are the CIS Retrieval Agent.

#Always use the ask_cis tool
#when answering security configuration questions.
#"""
#)


In [57]:
#threat_agent=create_agent(
 #   model=llm,
  #  tools=[lookup_cve],
   # system_prompt="""
#You are a CVE lookup assistant.

#Rules:
#- Call lookup_cve only once.
#- Use the user's keywords directly.
#- Return only the most relevant vulnerabilities.
#- Maximum 5 CVEs
#"""
#)


## Create the Cybersecurity Agent with Integrated Security Intelligence Tools


In [89]:
cyber_agent=create_agent(
    model=llm,

    tools=[
        ask_cis,
        lookup_cve,
        lookup_cisa_kev,
        lookup_attack,
        lookup_epss
    ],

    system_prompt="""
You are a cybersecurity assistant.

Use tools when needed:

ask_cis:
CIS hardening guidance.

lookup_cve:
CVE vulnerabilities.

lookup_cisa_kev:
actively exploited vulnerabilities.

lookup_attack:
MITRE attacker techniques.

lookup_epss:
CVE risk score.

Select only relevant tools.
Return concise answers.
"""
)


## Create the Validation and Report Generation Agents


In [90]:
validator_agent=create_agent(
    model=llm,
    tools=[],
    system_prompt="""
You are the Validation Agent.

Combine CIS guidance and CVE information.

Provide:
- accurate explanation
- security recommendations
- practical steps

Do not invent information.
"""
)

report_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="""
You are the Report Generation Agent.

Generate a professional cybersecurity report.

The report should contain:

1. Executive Summary

2. CIS Recommendations

3. Recent CVEs

4. Known Exploited Vulnerabilities (KEV)

5. MITRE ATT&CK Techniques

6. EPSS Scores

7. Risk Assessment

8. Final Recommendations

Use only the supplied information.
Do not invent facts.
"""
)

## Define the Shared State for the LangGraph Workflow


In [91]:
class CyberState(TypedDict):

    query: str

    use_cyber: bool

    cyber_output: str

    final: str


## Implement Planner, Cybersecurity Processing, and Validation Workflow Nodes


In [92]:
def planner_node(state):

    query = state["query"].lower()

    use_rag = False
    use_cve = False


    if "cis" in query or "benchmark" in query or "recommend" in query:
        use_rag = True

    if "cve" in query or "vulnerability" in query or "latest" in query:
        use_cve = True


    print("Planner Decision:")
    print({
        "use_rag": use_rag,
        "use_cve": use_cve
    })


    return {
        "query": query,
        "use_rag": use_rag,
        "use_cve": use_cve
    }





def cyber_node(state):

    print("---- CYBER NODE ----")

    response = cyber_agent.invoke(
        {
            "messages":[
                {
                    "role":"user",
                    "content":state["query"]
                }
            ]
        }
    )

    MAX_LENGTH = 2000

    cyber_output = response["messages"][-1].content

    cyber_output = cyber_output[:MAX_LENGTH]

    return {
        "cyber_output": cyber_output
    }


def validator_node(state):

    print("---- VALIDATOR NODE START ----")

    print("CVE DATA:")
    print(state.get("cves"))

    MAX_LENGTH = 2000

    prompt = f"""
Question

{state["query"]}

Cyber Agent Output

{state.get("cyber_output","")[:MAX_LENGTH]}

CVE Data

{str(state.get("cves",""))[:MAX_LENGTH]}

Validate the information.

Correct inconsistencies.

Produce one accurate cybersecurity response.
"""

    response = validator_agent.invoke(
        {
            "messages":[
                {
                    "role":"user",
                    "content": prompt
                }
            ]
        }
    )

    print("VALIDATOR RESPONSE:")
    print(response)

    return {
        "final": response["messages"][-1].content
    }




# Define Report Generation Node

This cell implements the Report Agent node.


In [93]:
def report_node(state):

    prompt = f"""
Generate a professional cybersecurity assessment.

Question:
{state['query']}

Validated Information:
{state['final']}
"""

    response = report_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        }
    )

    return {
        "final": response["messages"][-1].content
    }


# Build and Compile LangGraph Cybersecurity Workflow

This cell defines the workflow routing logic and builds the multi-agent cybersecurity pipeline.

In [94]:
def router(state):

    if state["use_rag"] and state["use_cve"]:
        return "both"

    elif state["use_rag"]:
        return "rag"

    elif state["use_cve"]:
        return "cve"

    else:
        return "none"


graph = StateGraph(CyberState)

graph.add_node(
    "planner",
    planner_node
)

graph.add_node(
    "cyber",
    cyber_node
)

graph.add_node(
    "validator",
    validator_node
)

graph.add_node(
    "report",
    report_node
)

graph.add_edge(
    START,
    "planner"
)

graph.add_conditional_edges(
    "planner",
    router,
    {
        "rag": "cyber",
        "cve": "cyber",
        "both": "cyber",
        "none": "validator"
    }
)

graph.add_edge(
    "cyber",
    "validator"
)

graph.add_edge(
    "validator",
    "report"
)

graph.add_edge(
    "report",
    END
)

app = graph.compile()

print("Cyber Security Agent ready")


Cyber Security Agent ready


# Execute Cybersecurity Agent Query and Generate Final Report

This cell runs the compiled LangGraph cybersecurity workflow with a user query.


In [95]:
result = app.invoke(
    {
        "query":"Show latest  vulnerabilities"
    }
)

print(result["final"])


Planner Decision:
{'use_rag': False, 'use_cve': True}
---- CYBER NODE ----
---- VALIDATOR NODE START ----
CVE DATA:
None
VALIDATOR RESPONSE:
{'messages': [HumanMessage(content='\nQuestion\n\nshow latest  vulnerabilities\n\nCyber Agent Output\n\nIt appears that the latest vulnerabilities are quite old, dating back to 2002-2004. It\'s likely that these vulnerabilities have been patched or fixed by now. If you\'re looking for the latest vulnerabilities, I would recommend using the lookup_cve function with a more recent keyword, such as "2022" or "2023".\n\nCVE Data\n\n\n\nValidate the information.\n\nCorrect inconsistencies.\n\nProduce one accurate cybersecurity response.\n', additional_kwargs={}, response_metadata={}, id='a6894b90-a0e8-412a-81fd-eced554511bb'), AIMessage(content='**Validation and Correction**\n\nUpon reviewing the provided information, I noticed that the Cyber Agent Output mentioned that the latest vulnerabilities date back to 2002-2004. However, the CVE Data is not prov

# Define Cybersecurity Test Query Dataset

This cell contains a collection of test queries used to evaluate the performance of the cybersecurity multi-agent workflow.

The test cases cover different scenarios:

- **CIS Benchmark Queries**
  - Windows hardening recommendations
  - Password policy configuration
  - Security control recommendations

- **CVE Vulnerability Queries**
  - Latest vulnerabilities
  - Critical CVE identification
  - Product-specific security issues

- **Combined Security Assessments**
  - CIS recommendations with vulnerability analysis
  - Security hardening guidance with threat intelligence
  - Enterprise application security reviews


In [96]:
test_queries = [
    "How do I disable SMBv1 according to the CIS benchmark?",

    "What does CIS recommend for Windows password policies?",

    "Show the latest vulnerabilities affecting Windows SMB.",

    "List the latest critical CVEs for OpenSSL.",

    "How can I secure Windows SMB against recent attacks?",

    "Recommend CIS controls for Apache HTTP Server and include recent CVEs.",

    "How should I harden OpenSSH based on CIS recommendations and current vulnerabilities?",

    "Explain how to secure Remote Desktop Protocol (RDP) and identify any known vulnerabilities.",

    "What CIS recommendations exist for Windows Firewall, and are there any recent Windows Firewall-related CVEs?",

    "Prepare a security assessment for Microsoft Exchange Server including CIS guidance and the latest vulnerabilities."
]


In [97]:
def run_test(query):

    print("\n" + "="*100)
    print("QUERY:")
    print(query)

    result = app.invoke(
        {
            "query": query
        }
    )

    print("\nFINAL REPORT:")
    print(result["final"])

    print("="*100)


# Run Automated Test Suite for Cybersecurity Workflow

This cell executes all predefined cybersecurity test queries sequentially.

For each test case, it:

- Displays the test case number.
- Sends the query through the complete LangGraph workflow.
- Executes planner, cybersecurity analysis, validation, and report generation steps.
- Prints the final cybersecurity assessment report.

In [98]:
for i, query in enumerate(test_queries, start=1):

    print(f"\n\n######## TEST CASE {i} ########")

    run_test(query)




######## TEST CASE 1 ########

QUERY:
How do I disable SMBv1 according to the CIS benchmark?
Planner Decision:
{'use_rag': True, 'use_cve': False}
---- CYBER NODE ----
---- VALIDATOR NODE START ----
CVE DATA:
None
VALIDATOR RESPONSE:
{'messages': [HumanMessage(content='\nQuestion\n\nhow do i disable smbv1 according to the cis benchmark?\n\nCyber Agent Output\n\nTo disable SMBv1 according to the CIS benchmark, you should set the following Group Policy:\n\nComputer Configuration\\Policies\\Administrative Templates\\MS Security Guide\\Configure SMB v1 server\n\nto Disabled.\n\nNote: This Group Policy path does not exist by default, so you will need to download and install the additional Group Policy template (SecGuide.admx/adml) from Microsoft.\n\nAlternatively, on Windows 10 R1709 or newer, you can also disable SMBv1 by setting the following Group Policy:\n\nComputer Configuration\\Policies\\Administrative Templates\\Network\\Lanman Workstation\\Disable SMBv1\n\nto Enabled.\n\nYou can 

# Define Test Case Classification Summary

This cell groups the test cases according to the expected workflow path.

Test categories:

- **RAG Only**
  - Queries that require CIS benchmark knowledge retrieval.
  - Examples:
    - CIS hardening recommendations
    - Security configuration guidance

- **CVE Only**
  - Queries focused on vulnerability identification and CVE information.
  - Examples:
    - Latest vulnerabilities
    - Critical CVE discovery

- **RAG + CVE**
  - Queries requiring both:
    - CIS security recommendations
    - Current vulnerability intelligence

This classification is used to evaluate whether the Planner Agent correctly identifies the required information sources and routes queries through the appropriate workflow.


In [100]:
test_summary = {
    "RAG Only": [
        1,
        2
    ],

    "CVE Only": [
        3,
        4
    ],

    "RAG + CVE": [
        5,
        6,
        7,
        8,
        9,
        10
    ]
}


test_summary


{'RAG Only': [1, 2], 'CVE Only': [3, 4], 'RAG + CVE': [5, 6, 7, 8, 9, 10]}